# 데이터셋 만들기 (코랩용)
위에서부터 한 칸씩 실행하세요. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**를 먼저 선택해야 합니다.

## 0. 설정 — 아래 두 줄만 내 정보로 바꾸세요

In [ ]:
GITHUB_USER = "내깃허브아이디"   # ← 바꾸기
REPO = "hard-negative-mllm"      # ← 저장소 이름이 다르면 바꾸기
EMAIL = "내이메일@example.com"     # ← 바꾸기 (GitHub 계정 이메일)

## 1. 구글 드라이브 연결 + GPU 확인
(난이도 계산 중 코랩이 끊겨도 드라이브에 중간 결과가 남습니다)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. 내 GitHub 저장소 내려받기
왼쪽 🔑(보안 비밀)에 `GITHUB_TOKEN`이라는 이름으로 토큰을 저장해 두어야 합니다.

In [ ]:
import os
from google.colab import userdata
TOKEN = userdata.get('GITHUB_TOKEN')
%cd /content
if not os.path.exists(REPO):
    !git clone https://{TOKEN}@github.com/{GITHUB_USER}/{REPO}.git
%cd /content/{REPO}
!git pull
!git config user.email "{EMAIL}"
!git config user.name "{GITHUB_USER}"
!pip install -q -r requirements.txt

In [ ]:
def push(msg):
    """결과물을 GitHub에 올리는 함수"""
    get_ipython().system("git add -A")
    get_ipython().system(f'git commit -m "{msg}"')
    get_ipython().system("git push origin main")

## 3. 1단계 — 원본 데이터 정리 (GPU 필요 없음, 수 분)

In [ ]:
!python scripts/01_prepare_data.py

In [ ]:
push("1단계: 원본 데이터 정리 결과 (ID 목록, 로그)")

## 4. 2단계 — 오답 난이도 측정 + 관문 A (GPU, 오래 걸림)
끊기면 이 칸을 다시 실행하세요. 드라이브에 저장된 곳부터 이어서 계산합니다.

In [ ]:
!mkdir -p /content/drive/MyDrive/hn_partial
!python scripts/02_score_difficulty.py --model smolvlm --partial /content/drive/MyDrive/hn_partial/partial_smolvlm.parquet

In [ ]:
import json
r = json.load(open('data/difficulty_report_smolvlm.json'))
for k, v in r['checks'].items():
    print(k, '통과' if v['pass'] else '미통과', v['value'])
print('전체 통과:', r['all_pass'])

In [ ]:
push("2단계: 오답 난이도 측정 및 관문 A 결과 (smolvlm)")

### ✋ 여기서 멈추고 D1 검수
`data/d1_audit.csv`와 `data/d1_audit_images/`를 내려받아 label 칸을 채운 뒤, GitHub에 `data/d1_audit.csv`를 다시 올리세요.
관문 A가 하나라도 **미통과**면 3단계로 가지 마세요.

## 5. 3단계 — 학습 세트 만들기 + 자동 검증

In [ ]:
!git pull
!python scripts/03_build_trainsets.py --model smolvlm

In [ ]:
push("3단계: smolvlm 학습 세트 105개 생성")
!git tag data-v1.0
!git push origin data-v1.0